# Cálculo de Elo
* Para previsões mais precisas de eventos desportivos é necessário o estabelecimento de métodos de cálculo. Um desses métodos é o estabelecimento de um "ranking" baseado em performances históricas, ou elo.
* A função deste notebook é calcular o elo para as equipes que participaram das edições da NBB de 2016 até 2025.

## Passos
1. Estabelecer um fator multiplicador (k) que diferencie o peso de cada temporada ou "fase" do torneio;
2. Definição de elo inicial para todas as equipes registradas;
3. Extrair e recálcular elo dos vencedores e perdedores após cada partida que participarem;
4. Criar um dataframe com o elo ao final de cada temporada, e o atual.

In [27]:
import pandas as pd                           # Biblioteca de dataframes
from collections import defaultdict           # Biblioteca para valor padrão automático
from pathlib import Path                      # Biblioteca para manipulação de caminhos de arquivos
import numpy as np                            # Biblioteca para cálculos matemáticos
import plotly.express as px                   # Biblioteca para visualização de dados

In [28]:
ROOT = Path.cwd().parent  # Caminho raiz do projeto

df_historico = pd.read_csv(ROOT / 'historico_nbb' / 'data' / 'historico_nbb.csv')  # Carrega o histórico de partidas

In [29]:
df_historico.head(15)

,Season,Casa,Placar_casa,Placar_visitante,Visitante
0,2016,Bauru,97,100,Flamengo
1,2016,Pinheiros,75,79,Vitória
2,2016,Minas,83,63,Caxias do Sul Basquete
3,2016,Franca,84,85,Flamengo
4,2016,Bauru,84,76,Macaé Basquete
5,2016,Pinheiros,80,74,Basquete Cearense
6,2016,Minas,83,85,Vasco da Gama
7,2016,Brasília,91,82,Caxias do Sul Basquete
8,2016,Franca,86,79,Macaé Basquete
9,2016,Brasília,95,72,Vasco da Gama


In [30]:
# ===========================================================
# Definição dos pesos do fator de multiplicação K
# ===========================================================

TIER_1 = {2025}
TIER_2 = {2024, 2023}
TIER_3 = {2020, 2021, 2022}
TIER_4 = {2016, 2017, 2018}

def get_k_factor(Season):
    # Quanto mais recente o histórico, maior o peso na variação de elo
    if Season in TIER_1: return 40
    if Season in TIER_2: return 32
    if Season in TIER_3: return 24
    if Season in TIER_4: return 16

In [31]:
# ===========================================================
# Fórmula de probabilidade de vitória
# ===========================================================
def win_probability(elo_casa, elo_fora, vantagem_casa=60):
    diff = (elo_casa + vantagem_casa) - elo_fora
    return 1 / (1 + 10 ** (-diff / 400))

df = df_historico.copy()  # Cria uma cópia do DataFrame para manipulação
df = df.dropna(subset=['Season', 'Casa', 'Placar_casa', 'Placar_visitante', 'Visitante'])  # Remove linhas com valores nulos nas colunas especificadas


# ===========================================================
# Inicialização do dicionário de elo
# ===========================================================
INITIAL_ELO = 1500  # Valor inicial de elo para cada time
elo_atual = defaultdict(lambda: INITIAL_ELO)  # Dicionário para armazenar os elos atuais dos times, com valor padrão inicial
elo_historico = []  # Lista para armazenar o histórico de elos após cada partida

for row in df.itertuples(index=False):  # Itera sobre cada linha do DataFrame
    casa, visitante = row.Casa, row.Visitante  # Extrai os nomes dos times da casa e visitante
    elo_casa, elo_visitante = elo_atual[casa], elo_atual[visitante]  # Obtém os elos atuais dos times

    prob_casa = win_probability(elo_casa, elo_visitante)  # Calcula a probabilidade de vitória do time da casa
    prob_visitante = 1 - prob_casa  # Calcula a probabilidade de vitória do time visitante

    # Resultado da partida
    if row.Placar_casa > row.Placar_visitante:
        resultado_casa, resultado_visitante = 1.0, 0.0  # Vitória do time da casa
    elif row.Placar_casa < row.Placar_visitante:
        resultado_casa, resultado_visitante = 0.0, 1.0  # Vitória do time visitante

    mov = abs(row.Placar_casa - row.Placar_visitante)  # Diferença de pontos entre os times
    
    if (row.Placar_casa == 20 and row.Placar_visitante == 0) or (row.Placar_casa == 0 and row.Placar_visitante == 20):
        mul = 1.0 # Se a partirda for um W.O
    else:
        mul = min(1.5, 1.0 + 0.5 * mov / (mov + 10)) # Assegura que o multiplicador não ultrapasse 1.5, ajustando a variação de elo com base na diferença de pontos

    K = get_k_factor(row.Season)  # Obtém o fator de multiplicação K com base na temporada

    elo_atual[casa]  = elo_casa + K * mul * (resultado_casa - prob_casa)  # Atualiza o elo do time da casa
    elo_atual[visitante] = elo_visitante + K * mul * (resultado_visitante - prob_visitante)  # Atualiza o elo do time visitante

    elo_historico.append((row.Season, casa, elo_atual[casa]))  # Armazena o elo atualizado do time da casa no histórico
    elo_historico.append((row.Season, visitante, elo_atual[visitante]))  # Armazena o elo atualizado do time visitante no histórico

# ===========================================================
# Criação do DataFrame de histórico de elos
# ===========================================================
df_elo = pd.DataFrame(elo_historico, columns=['Season', 'Time', 'Elo'])  # Cria um DataFrame com o histórico de elos
print(f"df_elo shape: {df_elo.shape}")  # Exibe o tamanho do DataFrame de histórico de elos

df_elo shape: (5568, 3)


In [32]:
# ===========================================================
# Observar o histórico de elos
# ===========================================================
rank = (df_elo.sort_values('Season')
        .groupby('Time')
        .tail(1)
        .sort_values('Elo', ascending=False)
        .head(15))

print(rank.to_string(index=False))  # Exibe os 15 times com maior elo na última temporada

 Season                   Time         Elo
   2025                 Franca 1910.205789
   2025              Pinheiros 1828.027639
   2025            Corinthians 1762.702554
   2025               Brasília 1760.525917
   2025               Flamengo 1708.818785
   2025                  Minas 1701.836481
   2025                   Mogi 1700.211726
   2025             Paulistano 1641.573135
   2025              UNIFACISA 1588.290432
   2025                  Bauru 1586.872178
   2025               São José 1531.172725
   2025      União Corinthians 1514.853433
   2025 Caxias do Sul Basquete 1499.711879
   2017                Vitória 1491.398754
   2024              São Paulo 1447.665266


In [33]:
df_elo['Time'].unique()

<StringArray>
[                 'Bauru',               'Flamengo',              'Pinheiros',
                'Vitória',                  'Minas', 'Caxias do Sul Basquete',
                 'Franca',         'Macaé Basquete',      'Basquete Cearense',
          'Vasco da Gama',               'Brasília',           'Campo Mourão',
          'L. Sorocabana',                   'Mogi',             'Paulistano',
       'Joinville / AABJ',               'Botafogo',               'São José',
            'Corinthians',            'VipTech CMB',              'UNIFACISA',
          'Pato Basquete',              'São Paulo',       'Cerrado Basquete',
      'União Corinthians',              'Rio Claro',               'Cruzeiro',
          'Basket Osasco']
Length: 28, dtype: str

In [34]:
# ===========================================================
# Gerar o arquivo CSV com o histórico de elos e
# com o elo atual pré-temporada 2026/2027
# ===========================================================

df_elo.to_csv(ROOT / 'historico_nbb' / 'data' / 'historico_elo.csv', index=False)  # Salva o DataFrame de histórico de elos em um arquivo CSV


elo_atual = df_elo[df_elo['Season'] == 2025].copy()  # Filtra o DataFrame para obter os elos atuais da temporada 2025
elo_atual.to_csv(ROOT / 'historico_nbb' / 'data' / 'elo_atual_2026.csv', index=False)  # Salva o DataFrame de elos atuais em um arquivo CSV

In [35]:
eloXtempo = df_elo.groupby(['Season', 'Time'])['Elo'].mean().reset_index()  # Agrupa o DataFrame por temporada e time, mantendo o elo correspondente

plot = px.line(eloXtempo, x='Season', y='Elo', color='Time', title='Evolução do Elo ao longo das temporadas')  # Cria um gráfico de linha para visualizar a evolução do elo ao longo das temporadas
plot.show()  

In [36]:
df_elo[df_elo['Time'] == 'Corinthians'].sort_values('Season', ascending=False)  # Filtra o DataFrame para exibir o histórico de elos do time Flamengo, ordenado por temporada

,Season,Time,Elo
5557,2025,Corinthians,1762.702554
5550,2025,Corinthians,1779.132603
5546,2025,Corinthians,1802.292993
5545,2025,Corinthians,1775.228137
5288,2025,Corinthians,1752.977755
...,...,...,...
1404,2018,Corinthians,1518.177312
1073,2018,Corinthians,1508.836634
1066,2018,Corinthians,1495.903371
1050,2018,Corinthians,1486.249573
